# Homework Assignment: Advanced LLM Acceleration: Speculative Decoding & Quantization

**Target hardware:** 1x NVIDIA H100 80GB

**Base model:** `Qwen/Qwen3-8B`

## Objective

Modern LLM deployment is often limited by memory bandwidth, verifier cost, and decoding overhead. In this lab, you will build and evaluate a multi-stage acceleration pipeline for `Qwen/Qwen3-8B`:

- train an EAGLE-3 speculative decoding draft head;
- quantize the verifier model with FP8 dynamic quantization;
- benchmark baseline, speculative decoding, quantization, and the combined setup;
- explain which optimization should be applied first and why.

The main question for the assignment:

**Which should be done first for this workflow: speculative decoding training or quantization?**

Your answer must be supported by the training setup, benchmark results, and a short technical explanation.

---

## References

- Speculators library: <https://github.com/vllm-project/speculators>
- Offline EAGLE-3 training tutorial: <https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline>
- FP8 dynamic quantization reference: <https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md>

---

## Required Library Versions

Use separate environments. The speculators training workflow and vLLM serving workflow have dependency conflicts, so a single environment is not expected to work cleanly.

| Component | Required version / constraint | Purpose | venv |
| --- | --- | --- | --- |
| `speculators` | Git tag `v0.5.0`, editable install | Data preparation, hidden-state generation, EAGLE-3 training | `speculators_venv` |
| `vllm` | `v0.20.0` | Serving and benchmark runtime | `vllm_venv` |
| `fastapi` | `<0.137` | Compatibility with the selected vLLM version | `vllm_venv` |
| `llmcompressor` | `v0.12.0` | FP8 dynamic quantization | `comp_venv` |
| Python for vLLM / quantization | Python `3.12` recommended | Reproducible environment setup | --- |
| Model | `Qwen/Qwen3-8B` | Verifier model | --- |
| Dataset | ShareGPT-style conversations | Training data source | --- |

Expected local environments:

- `speculators_venv`
- `vllm_venv`
- `comp_venv`

Do not submit the virtual environments.

Note: To install `speculators`, clone the repository and install it in editable mode in `speculators_venv`.

---

## Task 1: Environment & Data Orchestration

Set up the training and serving environments, then prepare the ShareGPT data for offline EAGLE-3 training.

Required work:

- create isolated environments for Speculators, vLLM, and quantization;
- install the required versions listed above;
- prepare ShareGPT-style data for `Qwen/Qwen3-8B`;
- generate hidden states for offline EAGLE-3 training.

Background:

Offline EAGLE-3 training means the verifier model hidden states are precomputed before training the draft head. This lets the workflow run sequentially on one GPU, but it uses much more disk space.

Online EAGLE-3 training computes hidden states during training. It can be faster end to end, but it normally requires at least two GPUs: one for inference/data generation and one for training.

Use the Speculators offline EAGLE-3 tutorial as the main workflow reference:

<https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline>

You need to implement an offline EAGLE-3 training pipeline.

Hints:

- Watch local disk usage. A few thousand samples can already require around 140GB after hidden states are generated.
- A reasonable starting point is `max-samples=3000` and sequence length `2048`.
- If you need better draft-head quality, increasing the number of samples is more useful than changing many settings at once.
- Use the `scripts/launch_vllm.py` script to run the vLLM server.

Question: Why do hidden states require much more disk space than the original text dataset?

Troubleshooting hints:

- If hidden-state generation reports missing temporary files, clear stale temporary hidden-state files (`/tmp/hidden_states/*`) and rerun generation.
- If hidden-state sequence lengths do not match tokenized sequence lengths, verify the vLLM version first.
- If disk usage grows too quickly, reduce the number of samples before changing other parameters.

---

## Task 1 — Answer

**Why do hidden states require much more disk space than the original text dataset?**

Because text is a *symbolic* representation (a few bytes per token), while a hidden state is a *dense high-dimensional float vector for every token, at several layers* — so each token expands from a handful of bytes into tens of kilobytes. Concretely, for this run (`Qwen/Qwen3-8B`, hidden size `4096`, layers saved `2, 18, 33, 36`, `bfloat16`):

- **Text:** one token ≈ a few bytes (raw characters), or `8` bytes stored as an `int64` token id.
- **Hidden states:** `4 layers × 4096 dims × 2 bytes = 32,768 bytes ≈ 32 KB` **per token**.

That is a **~4,000–8,000× blow-up per token**, driven by four multiplicative factors:

1. **Embedding dimension** — every token becomes a `4096`-wide vector instead of a single id.
2. **Multiple layers** — EAGLE-3 saves auxiliary hidden states from several layers (here 4: `2, 18, 33, 36`), multiplying the per-token cost.
3. **Float vs. symbol** — each value is a 2-byte float; text compresses far better and carries no per-dimension payload.
4. **Dense & uncompressed** — stored as raw `safetensors` tensors (no entropy coding), unlike text which is highly compressible.

**Measured in this run:** one sample `hs_0.safetensors` holds a `(1918, 4, 4096)` bf16 tensor = **62.9 MB** for a single conversation. Across `max-samples=3000` that totals **129 GB** of hidden states, versus only **~18 MB** for the preprocessed text dataset — matching the task hint that a few thousand samples can need ~140 GB.

This is the core trade-off of *offline* EAGLE-3 training: precomputing hidden states lets the pipeline run sequentially on one GPU, but pays for it in disk. It's also why the hints stress watching disk usage and reducing `max-samples` before changing other parameters.

---

# Task 2: Train an EAGLE-3 Draft Head

Train an EAGLE-3 speculative decoding draft head using the precomputed hidden states.

Required work:

- Use `Qwen/Qwen3-8B` as the verifier model for training.
- Save checkpoints under `output/checkpoints/`.
- Use the best checkpoint for serving and benchmarking.
- Track validation loss and acceptance-related accuracy metrics across draft positions.

Hints:

- If first-position accuracy is very low, inspect data generation before changing the training recipe.
- Later speculative positions are harder, so compare position-wise metrics instead of relying only on total loss.

Reference training result from one run:

| Metric | Value |
| --- | ---: |
| `val/loss_0_epoch` | `2.509` |
| `val/full_acc_0_epoch` | `0.463` |
| `val/cond_acc_0_epoch` | `0.463` |
| `val/loss_1_epoch` | `3.778` |
| `val/full_acc_1_epoch` | `0.181` |
| `val/cond_acc_1_epoch` | `0.364` |
| `val/loss_2_epoch` | `4.550` |
| `val/full_acc_2_epoch` | `0.069` |
| `val/cond_acc_2_epoch` | `0.320` |
| `val/loss_epoch` | `10.837` |
| Epoch | `4` |

Note: Epoch=4 means this is the 5th epoch, the index starts with 0.

Questions to answer:

1. What do `full_acc` and `cond_acc` measure in speculative decoding training?
2. Why does accuracy usually decrease for later speculative positions?
3. What would you change if the first-position accuracy is very low?

---

## Task 2 — Answers

**Our run reproduced the reference almost exactly** (`val/full_acc_0 = 0.463`, `val/loss_epoch = 10.842`, best at epoch 4), confirming the data + training pipeline is healthy. Best checkpoint: `output/checkpoints/checkpoint_best -> 4/`. Metric definitions below are taken from `compute_accuracy` in `speculators/src/speculators/models/eagle3/core.py`.

### 1. What do `full_acc` and `cond_acc` measure?

For each speculative step `k`, the draft's `argmax(logits)` is compared to the verifier's `argmax(targets)`. Both metrics share the **same numerator** — the count of tokens where the draft predicted *every* step from `0` through `k` correctly (predictions are chained via `logical_and` with `prev_correct`). They differ in the **denominator**:

- **`full_acc_k` = (chain correct through k) / (all tokens).** The *unconditional* probability that the first `k+1` drafted tokens are **all** accepted by the verifier — effectively the acceptance probability of the k-th speculative token. `Σ_k full_acc_k` approximates the expected **acceptance length** (mean tokens accepted per draft step).
- **`cond_acc_k` = (chain correct through k) / (tokens where steps 0..k-1 were all correct).** The *conditional* probability that step `k` is accepted **given the chain survived that far** — i.e. the per-step quality of the head at that depth.

At position 0 there is no prior step, so the denominators coincide and `full_acc_0 == cond_acc_0` (here `0.463`).

### 2. Why does accuracy decrease for later positions?

Two distinct effects — which is why `full_acc` falls faster than `cond_acc` (run: full `0.46 → 0.18 → 0.07` vs cond `0.46 → 0.36 → 0.32`):

- **`full_acc` drops mechanically:** it is a chained/joint quantity, so each extra position multiplies in another `<1` factor and it decays roughly geometrically even when each individual step is decent.
- **`cond_acc` still drops** because predicting further ahead is genuinely harder. The draft head does a test-time-training (TTT) rollout, conditioning on its *own* previously generated tokens/hidden states instead of ground truth — so approximation noise and errors **compound (exposure bias)**, and the verifier's next-token distribution gets flatter/more uncertain deeper into the future.

This is exactly why the hint says to compare **position-wise** metrics: the total `loss_epoch` (10.84) hides this structure; the per-position split shows the head is strong at depth 0 and degrades with depth, which is normal.

### 3. What would you change if first-position accuracy is very low?

A low `full_acc_0` / `cond_acc_0` signals a **fundamental misalignment, not a tuning problem** — position 0 is the easiest prediction, so failing it means the supervision is broken. Per the hint, **inspect data generation before changing the training recipe**, roughly in this order:

1. **Were hidden states actually generated and matched?** With `--on-missing skip`, missing cached states are *silently dropped*, so under-produced data → near-empty training → low accuracy. Verify `output/hidden_states/` is fully populated (use `--on-missing raise` to fail loudly).
2. **Layer-ID consistency** — the `--target-layer-ids` used to launch vLLM (`2, 18, 33, 36`) must match what training assumes; a mismatch feeds the head the wrong auxiliary hidden states.
3. **Token/hidden-state alignment** — off-by-one between a hidden state and its target token, or a **chat-template / tokenizer mismatch** between data prep and the verifier, corrupts supervision.
4. **Vocab mapping** (`t2d.npy` / `d2t.npy`) and `--draft-vocab-size` (32000) — a wrong draft↔target projection maps correct predictions to wrong token IDs.
5. **Right verifier & dtype** — confirm hidden states came from `Qwen/Qwen3-8B` and that `--hidden-states-dtype` matches what was written.

**Only after data is ruled out** would I touch the recipe (more data/epochs, learning rate, warmup). In our run `full_acc_0 = 0.463` matches the reference, so data generation is healthy and no change is needed.

---

## Task 3: Quantize the Verifier Model

Quantize `Qwen/Qwen3-8B` using FP8 dynamic quantization.

Required work:

- Use `llmcompressor`.
- Apply FP8 dynamic quantization to linear layers.
- Keep `lm_head` unquantized.
- Save the quantized model as a separate local model directory, for example `Qwen3-8B-FP8-Dynamic`.
- Do not overwrite the original model checkpoint.

Reference material:

<https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md>

Hints:

- Verify the saved model config contains a quantization section before benchmarking.
- Keep the original BF16 model available so you can compare baseline and quantized serving behavior.

Expected quantization properties:

| Property | Expected value |
| --- | --- |
| Quantization method | compressed tensors |
| Weight format | FP8 |
| Activation format | dynamic FP8 |
| Target modules | linear layers |
| Ignored module | `lm_head` |

Questions to answer:

1. Why is FP8 dynamic quantization useful for serving on H100?
2. Why might `lm_head` be excluded from quantization?
3. How can quantization affect speculative decoding acceptance rate?

---

## Task 3 — Answers

**Our run produced** `./Qwen3-8B-FP8-Dynamic/` (8.8 GB, ~half the BF16 size) with a valid `quantization_config`: `quant_method=compressed-tensors`, `format=float-quantized`, `weights=float8`, `input_activations=float8 (dynamic=True)`, `targets=[Linear]`, `ignore=[lm_head]` — matching every expected property, with the original BF16 model left untouched.

### 1. Why is FP8 dynamic quantization useful for serving on H100?

- **Native hardware support.** H100 (Hopper) has 4th-gen Tensor Cores with **native FP8 (E4M3) matmul**, so FP8 GEMMs run at roughly **2× the throughput of BF16** and are not merely emulated — the speedup is real on this GPU. (Older GPUs without FP8 tensor cores wouldn't benefit the same way.)
- **Halved memory + bandwidth.** Weights shrink from 2 bytes to 1 byte (our checkpoint went 16 GB → ~8.8 GB). That frees HBM for a **larger KV cache / more concurrent sequences / longer context**, and LLM decoding is memory-bandwidth-bound, so moving half as many weight bytes per token directly raises decode throughput.
- **Dynamic activations = no calibration, robust accuracy.** "Dynamic" means activation scales are computed **per-tensor at runtime** from the actual activations, instead of from a fixed calibration set. That makes it data-free and quick to produce (our run needed no dataset and finished in seconds), and it adapts to outlier activations, so accuracy degradation is typically negligible vs. BF16.
- **FP8's wide dynamic range.** Unlike INT8, FP8 is a floating-point format, so it tolerates the large activation outliers common in LLMs far better at 8 bits — a good accuracy/speed trade for serving.

### 2. Why might `lm_head` be excluded from quantization?

- **Accuracy sensitivity for little gain.** `lm_head` projects the hidden state onto the full vocabulary (here ~150k logits). Quantizing it injects noise **directly into the output logit distribution**, which can distort token probabilities and degrade generation quality — the most sensitive place to lose precision.
- **It's run once per token, not per layer.** The 36 transformer blocks dominate both compute and memory; `lm_head` is a single projection. Quantizing it saves little memory/throughput while risking the most, so the cost/benefit is poor.
- **Speculative-decoding correctness.** Acceptance is decided by comparing draft tokens against the **verifier's logits/sampled tokens**. Keeping `lm_head` in full precision keeps the verifier's token distribution as faithful as possible, protecting the accept/reject decision (see Q3). This is why the standard llmcompressor FP8 recipe ignores `lm_head` by default.

### 3. How can quantization affect speculative decoding acceptance rate?

Speculative decoding accepts a drafted token only when it agrees with what the **verifier** would have produced. Quantization perturbs the verifier's output distribution, which shifts that accept/reject boundary in two ways:

- **Verifier vs. draft mismatch.** If the verifier is quantized but the EAGLE-3 draft head was trained against the **BF16** verifier's hidden states/logits (as ours was), the draft is now predicting a slightly *different* target distribution. Tokens the draft proposes may no longer match the FP8 verifier's choices, **lowering the acceptance rate** even though each model alone is fine — a train/serve distribution mismatch.
- **Output quality vs. throughput trade.** FP8 noise can flip low-margin token decisions. Under greedy/low-temperature verification this directly reduces acceptance length; the FP8 GEMM speedup may still win on **net tokens/sec**, but acceptance rate itself usually drops slightly.
- **Mitigations.** Keep sensitive layers (e.g. `lm_head`) unquantized to preserve the logit distribution; ideally **regenerate hidden states from the FP8 verifier and retrain/calibrate the draft head against the quantized target** so draft and verifier agree again. This is exactly why Task 4 benchmarks BF16, FP8, and FP8 + speculative decoding separately — to measure the acceptance-rate impact rather than assume it.

---

## Task 4: Serve and Benchmark

Benchmark four configurations:

1. Baseline `Qwen/Qwen3-8B`
2. `Qwen/Qwen3-8B` with the trained EAGLE-3 draft head
3. FP8 dynamically quantized `Qwen3-8B`
4. FP8 dynamically quantized `Qwen3-8B` with the trained EAGLE-3 draft head

Required work:

- Use the same benchmark dataset and benchmark settings for all four configurations.
- Keep concurrency fixed across experiments.
- Disable prefix caching unless you intentionally study its effect.
- Use a fixed seed where possible.
- Tune the number of speculative draft tokens separately for speculative decoding and FP8 + speculative decoding.

Important:

The reference results used different numbers of speculative draft tokens for the unquantized speculative-decoding run and the FP8 + speculative-decoding run. Do not assume one value is optimal for both. Tune it yourself and justify the final choice using throughput, acceptance rate, acceptance length, and TPOT.

Hints:

- Start with a small number of speculative tokens, then increase only if the acceptance behavior supports it.
- Compare output token throughput first, then use TTFT, TPOT, and acceptance metrics to explain the result.
- If a setting produces more draft work but little accepted work, it is probably not the best setting.

Script for benchmarking:

```bash
vllm bench serve \
    --model Qwen/Qwen3-8B \
    --dataset-name hf \
    --max-concurrency 8 \
    --dataset-path philschmid/mt-bench \
    --num-prompts 80
```


Reference benchmark results:

| Configuration | Duration, s | Requests/s | Output tok/s | Total tok/s | Mean TTFT, ms | Mean TPOT, ms | Acceptance rate |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Baseline | `24.35` | `3.29` | `841.22` | `1090.87` | `576.17` | `7.28` | N/A |
| Speculative decoding | `16.27` | `4.92` | `1258.65` | `1632.19` | `78.17` | `5.76` | `22.48%` |
| FP8 quantization | `13.06` | `6.12` | `1566.56` | `2031.82` | `51.18` | `4.90` | N/A |
| FP8 + speculative decoding | `11.59` | `6.90` | `1766.55` | `2290.82` | `30.24` | `4.28` | `36.50%` |

Reference speculative decoding details:

| Configuration | Reference draft tokens | Acceptance length | Drafts | Draft tokens | Accepted tokens |
| --- | ---: | ---: | ---: | ---: | ---: |
| Speculative decoding | `2` | `1.45` | `14088` | `28176` | `6334` |
| FP8 + speculative decoding | `1` | `1.36` | `14954` | `14954` | `5458` |

Your exact numbers may differ, but the relative comparison should be explainable.

Questions to answer:

1. Why can speculative decoding improve throughput even when acceptance rate is not close to 100%?
2. How many speculative tokens are optimal for this setup? Explain using acceptance rate, acceptance length, and TPOT.

---

## Final Report Requirements

Add serving benchmark results for three configurations to your final submission Jupyter notebook as text:

- speculative decoding;
- FP8 quantization;
- FP8 + speculative decoding.

---

## Scoring Rubric

Scores are based on `Output token throughput (tok/s)` from `vllm bench serve`.
Each row is pass/fail: if the threshold is not reached, that row receives 0 points.

| Requirement | Passing threshold | Points |
| --- | ---: | ---: |
| Speculative decoding result with trained EAGLE-3 draft head | `> 1250 tok/s` | 25 |
| FP8 dynamic quantization result | `> 1550 tok/s` | 10 |
| Best combined FP8 + speculative decoding result, with draft-token tuning and explanation | `> 1750 tok/s` | 15 |
| **Total** |  | **50** |




This is an example of the output we expect to be submitted. This is the result we get from the baseline model out of the box:
```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  24.35     
Total input tokens:                      6078      
Total generated tokens:                  20480     
Request throughput (req/s):              3.29      
Output token throughput (tok/s):         841.22    
Peak output token throughput (tok/s):    1144.00   
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          1090.87   
---------------Time to First Token----------------
Mean TTFT (ms):                          576.17    
Median TTFT (ms):                        46.05     
P99 TTFT (ms):                           5677.04   
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          7.28      
Median TPOT (ms):                        7.01      
P99 TPOT (ms):                           11.66     
---------------Inter-token Latency----------------
Mean ITL (ms):                           7.28      
Median ITL (ms):                         7.00      
P99 ITL (ms):                            7.68      
==================================================

```

Speculative decoding benchmark results:
```
TODO
```

FP8 quantization benchmark results:

```
TODO
```

FP8 + speculative decoding benchmark results:

```
TODO
```